In [ ]:
import os, numpy as np, pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import confusion_matrix, classification_report

ROOT = "/kaggle/input/datasets/nghiala222/processed-split-2level/processed_split_2level"

TRAIN_ROOT = os.path.join(ROOT, "train")
VAL_ROOT   = os.path.join(ROOT, "val")
TEST_ROOT  = os.path.join(ROOT, "test")

IMG_SIZE = (224, 224)
BATCH = 32
SEED = 42
AUTOTUNE = tf.data.AUTOTUNE

In [ ]:
def scan_split(split_root):
    rows = []
    for fruit in os.listdir(split_root):
        fruit_dir = os.path.join(split_root, fruit)
        if not os.path.isdir(fruit_dir):
            continue
        for quality in ["fresh", "rotten"]:
            q_dir = os.path.join(fruit_dir, quality)
            if not os.path.isdir(q_dir):
                continue
            for fn in os.listdir(q_dir):
                path = os.path.join(q_dir, fn)
                if os.path.isfile(path):
                    rows.append({
                        "path": path,
                        "fruit": fruit,
                        "quality": quality,
                        "label": f"{fruit}_{quality}"
                    })
    return pd.DataFrame(rows)

train_df = scan_split(TRAIN_ROOT)
val_df   = scan_split(VAL_ROOT)
test_df  = scan_split(TEST_ROOT)

print("Scanned Train/Val/Test:", len(train_df), len(val_df), len(test_df))
print(train_df.head())

In [ ]:
def tf_can_decode(path):
    try:
        b = tf.io.read_file(path)
        _ = tf.image.decode_image(b, channels=3, expand_animations=False)
        return True
    except:
        return False

def filter_df_by_tf_decode(df, name="df"):
    ok_paths = []
    bad_paths = []
    for p in df["path"].tolist():
        if tf_can_decode(p):
            ok_paths.append(p)
        else:
            bad_paths.append(p)

    df2 = df[df["path"].isin(ok_paths)].reset_index(drop=True)

    print(f"[{name}] TF-decode filter: {len(df)} -> {len(df2)} | removed: {len(df)-len(df2)}")
    if len(bad_paths) > 0:
        print(f"[{name}] Example bad files (first 5):")
        for bp in bad_paths[:5]:
            print("  -", bp)

    return df2

train_df = filter_df_by_tf_decode(train_df, "train")
val_df   = filter_df_by_tf_decode(val_df, "val")
test_df  = filter_df_by_tf_decode(test_df, "test")

In [ ]:
class_names = sorted(train_df["label"].unique())
label_to_idx = {c:i for i,c in enumerate(class_names)}
idx_to_label = {i:c for c,i in label_to_idx.items()}
NUM_CLASSES = len(class_names)

print("NUM_CLASSES:", NUM_CLASSES)
print("Sample classes:", class_names[:10])

In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomBrightness(0.1),
], name="augmentation")

def load_img(path):
    img_bytes = tf.io.read_file(path)
    img = tf.image.decode_image(img_bytes, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32) / 255.0  # Normalize [0,1]
    return img

def make_ds(df, training=False):
    paths = df["path"].values
    y = df["label"].map(label_to_idx).values.astype(np.int32)

    ds = tf.data.Dataset.from_tensor_slices((paths, y))
    ds = ds.map(lambda p, yy: (load_img(p), yy), num_parallel_calls=AUTOTUNE)

    if training:
        ds = ds.shuffle(2000, seed=SEED, reshuffle_each_iteration=True)
        ds = ds.map(lambda x, yy: (data_augmentation(x, training=True), yy),
                    num_parallel_calls=AUTOTUNE)

    ds = ds.batch(BATCH).prefetch(AUTOTUNE)
    return ds

train_ds = make_ds(train_df, training=True)
val_ds   = make_ds(val_df, training=False)
test_ds  = make_ds(test_df, training=False)

In [ ]:
def build_cnn(num_classes):
    model = keras.Sequential([
        layers.Input(shape=(224,224,3)),

        layers.Conv2D(32, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(),

        layers.Conv2D(64, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(),

        layers.Conv2D(128, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(),

        layers.Dropout(0.5),
        layers.Flatten(),
        layers.Dense(256, activation="relu"),
        layers.Dense(num_classes, activation="softmax"),
    ], name="CNN_Baseline")

    model.compile(
        optimizer=keras.optimizers.Adam(1e-4),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

cnn = build_cnn(NUM_CLASSES)
cnn.summary()

In [ ]:
cnn_callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
    keras.callbacks.ModelCheckpoint("cnn_best.keras", monitor="val_accuracy", save_best_only=True),
]

history_cnn = cnn.fit(
    train_ds,
    validation_data=val_ds,
    epochs=3,
    callbacks=cnn_callbacks
)

cnn.save("/kaggle/working/cnn_fruit_quality.keras")
print("✅ Saved CNN: /kaggle/working/cnn_fruit_quality.keras")

In [ ]:
y_true = np.concatenate([y.numpy() for x,y in test_ds], axis=0)
y_prob = cnn.predict(test_ds, verbose=0)
y_pred = np.argmax(y_prob, axis=1)

print("CNN Confusion Matrix:\n", confusion_matrix(y_true, y_pred))
print("\nCNN Report:\n", classification_report(y_true, y_pred, target_names=class_names))

In [ ]:
def build_mobilenet(num_classes, lr=1e-4):
    base = keras.applications.MobileNet(
        input_shape=(224,224,3),
        include_top=False,
        weights="imagenet"
    )
    base.trainable = False

    inputs = keras.Input(shape=(224,224,3))
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = keras.Model(inputs, outputs, name="MobileNet_TL")
    model.compile(
        optimizer=keras.optimizers.Adam(lr),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model, base

mb_model, mb_base = build_mobilenet(NUM_CLASSES, lr=1e-4)
mb_model.summary()

In [ ]:
def build_mobilenet(num_classes, lr=1e-4):
    base = keras.applications.MobileNet(
        input_shape=(224,224,3),
        include_top=False,
        weights="imagenet"
    )
    base.trainable = False

    inputs = keras.Input(shape=(224,224,3))
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = keras.Model(inputs, outputs, name="MobileNet_TL")
    model.compile(
        optimizer=keras.optimizers.Adam(lr),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model, base

mb_model, mb_base = build_mobilenet(NUM_CLASSES, lr=1e-4)
mb_model.summary()

In [ ]:
mb_callbacks1 = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
    keras.callbacks.ModelCheckpoint("mobilenet_stage1.keras", monitor="val_accuracy", save_best_only=True),
]

history_mb1 = mb_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=3,
    callbacks=mb_callbacks1
)

In [ ]:
mb_base.trainable = True

fine_tune_at = 100  # chỉnh 80/120 nếu cần
for layer in mb_base.layers[:fine_tune_at]:
    layer.trainable = False

mb_model.compile(
    optimizer=keras.optimizers.Adam(1e-5),   # lr nhỏ khi fine-tune
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

mb_callbacks2 = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True),
    keras.callbacks.ModelCheckpoint("mobilenet_stage2.keras", monitor="val_accuracy", save_best_only=True),
]

history_mb2 = mb_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=3,
    callbacks=mb_callbacks2
)

mb_model.save("/kaggle/working/mobilenet_fruit_quality.keras")
print("✅ Saved MobileNet: /kaggle/working/mobilenet_fruit_quality.keras")

In [ ]:
y_prob_mb = mb_model.predict(test_ds, verbose=0)
y_pred_mb = np.argmax(y_prob_mb, axis=1)

print("MobileNet Confusion Matrix:\n", confusion_matrix(y_true, y_pred_mb))
print("\nMobileNet Report:\n", classification_report(y_true, y_pred_mb, target_names=class_names))

In [ ]:
def decode_label(label_str):
    fruit, quality = label_str.rsplit("_", 1)
    return fruit, quality

def predict_one(img_path, use_mobilenet=True):
    img = keras.utils.load_img(img_path, target_size=IMG_SIZE)
    arr = keras.utils.img_to_array(img).astype("float32") / 255.0
    arr = np.expand_dims(arr, 0)

    model = mb_model if use_mobilenet else cnn
    prob = model.predict(arr, verbose=0)[0]
    idx = int(np.argmax(prob))
    conf = float(np.max(prob))
    lab = idx_to_label[idx]

    fruit, quality = decode_label(lab)
    return fruit, quality, conf

sample = test_df.iloc[0]["path"]
print("Sample:", sample)
print("CNN pred:", predict_one(sample, use_mobilenet=False))
print("MB  pred:", predict_one(sample, use_mobilenet=True))

In [ ]:
import json
with open("/kaggle/working/class_names.json", "w") as f:
    json.dump(class_names, f)